# QBronze 1.4 — Entrelazamiento y protocolos

Este cuaderno desarrolla los ejercicios centrales sobre estados entrelazados, codificación superdensa y teleportación cuántica.

Se trabaja con amplitudes reales y con vectores de estado en la base computacional. La notación será explícita:

$$
|ab\rangle=|a\rangle\otimes |b\rangle.
$$

Para tres qubits usaremos el orden lógico:

$$
|\text{primer qubit},\text{segundo qubit},\text{tercer qubit}\rangle.
$$




In [ ]:
import numpy as np
from math import sqrt, pi, cos, sin, log2, floor
from collections import Counter

np.set_printoptions(precision=6, suppress=True)

ket0 = np.array([1.0, 0.0])
ket1 = np.array([0.0, 1.0])

I = np.eye(2)
X = np.array([[0.0, 1.0], [1.0, 0.0]])
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
H = (1 / sqrt(2)) * np.array([[1.0, 1.0], [1.0, -1.0]])

def tensor(*vectors_or_matrices):
    """Producto tensorial en el orden escrito: A ⊗ B ⊗ C."""
    result = np.array([1.0])
    for item in vectors_or_matrices:
        result = np.kron(result, item)
    return result

def norm2(state):
    """Norma al cuadrado: suma |amplitud|^2."""
    return float(np.vdot(state, state).real)

def probs(state):
    """Probabilidades de medición en la base computacional."""
    return np.abs(state) ** 2

def basis_state(bitstring):
    """Vector base |bitstring⟩ en orden lógico de izquierda a derecha."""
    n = len(bitstring)
    index = int(bitstring, 2)
    v = np.zeros(2**n)
    v[index] = 1.0
    return v

def bitstrings(n):
    return [format(i, f"0{n}b") for i in range(2**n)]

def show_state(state, n=None, tol=1e-10):
    """Representación textual compacta de un vector de estado."""
    if n is None:
        n = int(round(log2(len(state))))
    terms = []
    for amp, bits in zip(state, bitstrings(n)):
        if abs(amp) > tol:
            terms.append(f"{amp:+.6g}|{bits}⟩")
    return " ".join(terms) if terms else "0"

def sample_counts(state, shots=1024, seed=7, qiskit_order=False):
    """Muestreo de una distribución de medición. Si qiskit_order=True, invierte las etiquetas."""
    rng = np.random.default_rng(seed)
    p = probs(state)
    n = int(round(log2(len(state))))
    outcomes = rng.choice(2**n, p=p/p.sum(), size=shots)
    labels = []
    for i in outcomes:
        bits = format(i, f"0{n}b")
        labels.append(bits[::-1] if qiskit_order else bits)
    return dict(sorted(Counter(labels).items()))

def is_valid_quantum_state(v, tol=1e-9):
    return abs(norm2(np.asarray(v, dtype=float)) - 1.0) < tol


def cnot_matrix(control=0, target=1, n=2):
    size = 2**n
    M = np.zeros((size, size))
    for i, bits in enumerate(bitstrings(n)):
        b = list(bits)
        if b[control] == '1':
            b[target] = '0' if b[target] == '1' else '1'
        j = int(''.join(b), 2)
        M[j, i] = 1.0
    return M

def apply_single_qubit_gate(state, gate, qubit, n):
    ops = [gate if k == qubit else I for k in range(n)]
    return tensor(*ops) @ state

def coefficient_matrix_two_qubits(state):
    """Convierte [a00,a01,a10,a11] en matriz [[a00,a01],[a10,a11]]."""
    return np.array([[state[0], state[1]], [state[2], state[3]]])

def is_entangled_two_qubit(state, tol=1e-9):
    """Para dos qubits puros: está entrelazado si la matriz de coeficientes tiene rango 2."""
    M = coefficient_matrix_two_qubits(np.asarray(state, dtype=float))
    return np.linalg.matrix_rank(M, tol=tol) > 1

def density_matrix(state):
    state = np.asarray(state, dtype=complex)
    return np.outer(state, np.conjugate(state))

def partial_trace_second_qubit(rho):
    """Traza parcial sobre el segundo qubit de un sistema de dos qubits."""
    rho4 = rho.reshape(2, 2, 2, 2)  # a,b,a',b'
    return np.einsum('abcb->ac', rho4)

## 1. Preparar un estado de Bell

Partimos de

$$
|00\rangle.
$$

Primero aplicamos $H$ al primer qubit:

$$
(H\otimes I)|00\rangle
=
\frac{|0\rangle+|1\rangle}{\sqrt{2}}\otimes |0\rangle
=
\frac{|00\rangle+|10\rangle}{\sqrt{2}}.
$$

Luego aplicamos CNOT con el primer qubit como control y el segundo como objetivo:

$$
|00\rangle\mapsto |00\rangle,
\qquad
|10\rangle\mapsto |11\rangle.
$$

Por tanto:

$$
|00\rangle
\xrightarrow{H\otimes I}
\frac{|00\rangle+|10\rangle}{\sqrt{2}}
\xrightarrow{CNOT}
\frac{|00\rangle+|11\rangle}{\sqrt{2}}.
$$

Este estado no puede escribirse como producto de dos estados de un qubit; por eso está entrelazado.

In [ ]:
state = basis_state("00")
state = tensor(H, I) @ state
state = cnot_matrix(control=0, target=1, n=2) @ state
print("Estado de Bell:", show_state(state, n=2))
print("¿Está entrelazado?", is_entangled_two_qubit(state))
print("Matriz de coeficientes:")
print(coefficient_matrix_two_qubits(state))

## 2. Circuito GHZ de tres qubits

Consideramos el circuito:

1. Iniciar en $|000\rangle$.
2. Aplicar $H$ al primer qubit.
3. Aplicar CNOT del primer qubit al segundo.
4. Aplicar CNOT del primer qubit al tercero.

Paso a paso:

$$
|000\rangle
\xrightarrow{H_1}
\frac{|000\rangle+|100\rangle}{\sqrt{2}}.
$$

Primer CNOT:

$$
|100\rangle\mapsto |110\rangle.
$$

Segundo CNOT:

$$
|110\rangle\mapsto |111\rangle.
$$

Resultado:

$$
\frac{|000\rangle+|111\rangle}{\sqrt{2}}.
$$

Al medir muchas veces, sólo deben aparecer $000$ y $111$, aproximadamente mitad y mitad.

In [ ]:
state = basis_state("000")
state = apply_single_qubit_gate(state, H, qubit=0, n=3)
state = cnot_matrix(control=0, target=1, n=3) @ state
state = cnot_matrix(control=0, target=2, n=3) @ state
print("Estado GHZ:", show_state(state, n=3))
print("Conteos simulados:", sample_counts(state, shots=1000, seed=10))

## 3. Identificar estados entrelazados de dos qubits

Para un estado puro de dos qubits

$$
|\psi\rangle=a_{00}|00\rangle+a_{01}|01\rangle+a_{10}|10\rangle+a_{11}|11\rangle,
$$

formamos la matriz de coeficientes

$$
A=\begin{pmatrix}
a_{00}&a_{01}\\
a_{10}&a_{11}
\end{pmatrix}.
$$

El estado es separable si la matriz tiene rango 1. Es entrelazado si tiene rango 2.

Ejemplos típicos entrelazados:

$$
\frac{|01\rangle-|10\rangle}{\sqrt{2}},
\qquad
\frac{|00\rangle-|11\rangle}{\sqrt{2}}.
$$

Ejemplo no entrelazado:

$$
\frac{|00\rangle+|01\rangle+|10\rangle+|11\rangle}{2}
=\left(\frac{|0\rangle+|1\rangle}{\sqrt{2}}\right)
\otimes
\left(\frac{|0\rangle+|1\rangle}{\sqrt{2}}\right).
$$

In [ ]:
states = {
    "(|01>-|10>)/sqrt(2)": (basis_state("01") - basis_state("10")) / sqrt(2),
    "(|00>-|11>)/sqrt(2)": (basis_state("00") - basis_state("11")) / sqrt(2),
    "(|00>+|01>+|10>+|11>)/2": (basis_state("00") + basis_state("01") + basis_state("10") + basis_state("11")) / 2,
    "(|0>+|1>)/sqrt(2) no es de dos qubits": None,
}

for name, v in states.items():
    if v is None:
        print(f"{name}: no se evalúa como estado de dos qubits")
    else:
        print(f"{name:38s}  entrelazado = {is_entangled_two_qubit(v)}")

## 4. Leer correlaciones en un estado de tres qubits

Sea

$$
|\psi\rangle=\frac{|000\rangle+|001\rangle+|111\rangle}{\sqrt3}.
$$

Los únicos resultados posibles al medir en la base computacional son:

$$
000,
\quad
001,
\quad
111.
$$

Cada uno tiene probabilidad

$$
\left(\frac1{\sqrt3}\right)^2=\frac13.
$$

Consecuencias:

- Si el primer qubit se mide como $0$, los resultados compatibles son $000$ y $001$. En ambos, el segundo qubit es $0$.
- Si el primer qubit se mide como $1$, el único resultado compatible es $111$. Entonces el tercer qubit es $1$ con probabilidad 1.

In [ ]:
psi = (basis_state("000") + basis_state("001") + basis_state("111")) / sqrt(3)
print("Estado:", show_state(psi, n=3))

p = probs(psi)
for bits, prob in zip(bitstrings(3), p):
    if prob > 1e-12:
        print(bits, prob)

def conditional_probability(state, condition, event):
    n = int(round(log2(len(state))))
    p_total_condition = 0.0
    p_event_and_condition = 0.0
    for bits, prob in zip(bitstrings(n), probs(state)):
        if condition(bits):
            p_total_condition += prob
            if event(bits):
                p_event_and_condition += prob
    return p_event_and_condition / p_total_condition

print("P(segundo=0 | primero=0) =", conditional_probability(psi, lambda b: b[0]=='0', lambda b: b[1]=='0'))
print("P(tercero=1 | primero=1) =", conditional_probability(psi, lambda b: b[0]=='1', lambda b: b[2]=='1'))
print("P(tercero=1 | primero=0) =", conditional_probability(psi, lambda b: b[0]=='0', lambda b: b[2]=='1'))

## 5. Recursos en codificación superdensa

En codificación superdensa, un par entrelazado permite transmitir 2 bits clásicos mediante el envío físico de 1 qubit, siempre que el par entrelazado haya sido compartido antes.

Supongamos que el mensaje es `1302`, y que la codificación de cada dígito es:

$$
0\mapsto 00,
\quad
1\mapsto 01,
\quad
2\mapsto 10,
\quad
3\mapsto 11.
$$

El mensaje tiene 4 dígitos. Cada dígito corresponde a 2 bits. Por tanto se usa el protocolo 4 veces.

Cada uso consume 1 par entrelazado. Entonces se necesitan:

$$
4\text{ pares entrelazados}=8\text{ qubits en total}.
$$

No se reutiliza el mismo par, porque el entrelazamiento se consume durante el protocolo.

In [ ]:
message = "1302"
encoding = {"0": "00", "1": "01", "2": "10", "3": "11"}
encoded = [encoding[d] for d in message]
print("mensaje:", message)
print("bloques de 2 bits:", encoded)
print("número de usos del protocolo:", len(encoded))
print("pares entrelazados necesarios:", len(encoded))
print("qubits totales en esos pares:", 2 * len(encoded))

## 6. Operaciones de Alice en codificación superdensa

Usamos el estado inicial

$$
|\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt{2}}.
$$

Una convención común es:

$$
00\mapsto I,
\qquad
01\mapsto X,
\qquad
10\mapsto Z,
\qquad
11\mapsto XZ.
$$

Si Alice quiere enviar `01`, aplica $X$ a su qubit.

Verificación:

$$
(X\otimes I)|\Phi^+\rangle
=\frac{|10\rangle+|01\rangle}{\sqrt{2}}.
$$

Bob, al decodificar con CNOT y luego $H$ sobre el primer qubit, obtiene los bits clásicos correspondientes.

In [ ]:
phi_plus = (basis_state("00") + basis_state("11")) / sqrt(2)
encoded_01 = tensor(X, I) @ phi_plus
print("Estado después de X en el qubit de Alice:", show_state(encoded_01, n=2))

# Decodificación: CNOT primer->segundo, luego H en el primero.
decoded = tensor(H, I) @ (cnot_matrix(control=0, target=1, n=2) @ encoded_01)
print("Estado decodificado:", show_state(decoded, n=2))
print("Resultado medido con probabilidad 1:", max(zip(bitstrings(2), probs(decoded)), key=lambda x: x[1]))

## 7. Por qué Bob no puede recuperar el mensaje si pierde su qubit

En codificación superdensa, Bob necesita dos cosas:

1. El qubit que Alice le envía.
2. Su propio qubit, que estaba entrelazado con el de Alice.

Si Bob pierde su qubit, sólo conserva un subsistema. Ese subsistema por sí solo no contiene información suficiente para distinguir todos los mensajes.

Matemáticamente, para los cuatro mensajes, el estado reducido del qubit de Alice es el mismo:

$$
\rho_A=\frac12 I.
$$

Si los estados reducidos son iguales, ninguna medición local sobre ese único qubit puede recuperar el mensaje completo.

In [ ]:
ops = {
    "00: I": I,
    "01: X": X,
    "10: Z": Z,
    "11: XZ": X @ Z,
}

for label, op in ops.items():
    state = tensor(op, I) @ phi_plus
    rho = density_matrix(state)
    rho_alice = partial_trace_second_qubit(rho)
    print(label)
    print(np.real_if_close(rho_alice))
    print("---")

## 8. Operaciones condicionadas por mediciones clásicas

Considérese la lógica:

1. Iniciar dos qubits en $|00\rangle$.
2. Aplicar $X$ a $q[0]$, de modo que $q[0]=1$.
3. Medir $q[0]$ en $c[0]$, por lo que $c[0]=1$.
4. Aplicar $H$ a $q[1]$ sólo si el registro clásico completo vale 0.

Como el registro clásico ya contiene $c[0]=1$, la condición `c == 0` no se cumple. Por tanto no se aplica $H$ a $q[1]$.

El estado medido permanece con $q[1]=0$ y $q[0]=1$. La cadena impresa tipo $c[1]c[0]$ es `01`.

In [ ]:
# Simulación clásica del flujo lógico descrito.
q0 = 0
q1 = 0
q0 = 1 - q0        # X en q0
c0 = q0            # medición de q0 en c0
c1 = 0             # aún no medido
c_register_value = c0 + 2*c1

if c_register_value == 0:
    # Se habría aplicado H a q1, pero no entra aquí.
    q1_state = "superposición"
else:
    q1_state = q1

c1 = 0 if q1_state != "superposición" else "0 o 1"
print("valor del registro clásico después de medir q0:", c_register_value)
print("¿se aplica H a q1?", c_register_value == 0)
print("cadena final esperada c1c0:", "01")

## 9. Afirmaciones conceptuales sobre protocolos

Afirmaciones correctas:

- En codificación superdensa se pueden transmitir 2 bits clásicos mediante el envío físico de 1 qubit, usando además un par entrelazado previamente compartido.

Afirmaciones incorrectas:

- El entrelazamiento no permite comunicación más rápida que la luz.
- La teleportación no crea múltiples copias del mismo qubit; no viola el teorema de no clonación.
- No existe un único estado entrelazado; existen muchos estados entrelazados.
- En codificación superdensa, las operaciones que dependen del mensaje las aplica Alice, no Bob.

In [ ]:
statements = {
    "Entrelazamiento permite comunicación más rápida que la luz": False,
    "En codificación superdensa Alice codifica el mensaje": True,
    "Hay un único estado entrelazado": False,
    "Teleportación crea copias múltiples": False,
    "Codificación superdensa envía 2 bits clásicos usando 1 qubit transmitido y entrelazamiento previo": True,
}

for s, val in statements.items():
    print(f"{val!s:5s}  {s}")

## 10. Correcciones de Bob en teleportación cuántica

Sea el estado desconocido de Alice

$$
|\psi\rangle=\alpha|0\rangle+\beta|1\rangle.
$$

Después de las operaciones de Alice y de su medición, Bob puede quedar en uno de cuatro estados:

$$
\begin{array}{c|c|c}
\text{medición de Alice} & \text{estado de Bob antes de corrección} & \text{corrección}\\
\hline
00 & \alpha|0\rangle+\beta|1\rangle & I\\
01 & \alpha|1\rangle+\beta|0\rangle & X\\
10 & \alpha|0\rangle-\beta|1\rangle & Z\\
11 & \alpha|1\rangle-\beta|0\rangle & ZX
\end{array}
$$

Si Alice informa `10`, Bob aplica $Z$.

In [ ]:
alpha = 0.6
beta = sqrt(1 - alpha**2)
psi = np.array([alpha, beta])

bob_before = {
    "00": np.array([alpha, beta]),
    "01": np.array([beta, alpha]),
    "10": np.array([alpha, -beta]),
    "11": np.array([-beta, alpha]),  # equivale a aplicar X y luego Z hasta fase global según convención
}
correction = {
    "00": I,
    "01": X,
    "10": Z,
    "11": Z @ X,
}

for outcome in ["00", "01", "10", "11"]:
    corrected = correction[outcome] @ bob_before[outcome]
    # Comparamos hasta fase global: si difiere por -1, físicamente es el mismo estado.
    same_direct = np.allclose(corrected, psi)
    same_global_minus = np.allclose(corrected, -psi)
    print(outcome, "antes:", bob_before[outcome], "después:", corrected, "recupera ψ:", same_direct or same_global_minus)

## 11. Interpretación del vector de estado tras la medición de Alice

En una simulación de vector de estado con tres qubits, los estados base se ordenan como

$$
|000\rangle, |001\rangle, |010\rangle, |011\rangle,
|100\rangle, |101\rangle, |110\rangle, |111\rangle.
$$

Si todas las entradas son cero excepto las últimas dos, entonces el estado vive en el subespacio generado por

$$
|110\rangle, |111\rangle.
$$

En ambos casos, los dos primeros bits son `11`. Por tanto, la medición de Alice fue `11`.

Al final del protocolo, Bob conserva un único qubit: el qubit que contiene el estado teleportado.

In [ ]:
# Vector con entradas no nulas sólo en |110> y |111>.
state = np.zeros(8)
state[6] = 0.6
state[7] = 0.8
print("Estado:", show_state(state, n=3))
nonzero = [bits for bits, amp in zip(bitstrings(3), state) if abs(amp) > 1e-12]
print("bases no nulas:", nonzero)
print("primeros dos bits en todas ellas:", sorted(set(bits[:2] for bits in nonzero)))
print("qubits finales de Bob:", 1)

## 12. Resumen operativo del módulo

Ideas esenciales:

$$
|\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt{2}}
$$

es un recurso entrelazado.

La codificación superdensa usa:

$$
\text{1 qubit transmitido} + \text{1 par entrelazado previo}
\quad\Longrightarrow\quad
\text{2 bits clásicos comunicados}.
$$

La teleportación usa:

$$
\text{2 bits clásicos transmitidos} + \text{1 par entrelazado previo}
\quad\Longrightarrow\quad
\text{1 qubit teleportado}.
$$

El entrelazamiento se consume como recurso. La teleportación no clona el estado y no permite comunicación superlumínica.